# Credit Card Fraud Detection

### Libraries and Dataset Import
We import the required libraries and load the training dataset. A quick look at the first rows gives an overview of the available features.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler, QuantileTransformer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, roc_auc_score, average_precision_score,
                              ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
                              precision_recall_curve, f1_score)
import torch
import torch.nn.functional as F
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset


In [ ]:
df = pd.read_csv("archive/fraudTrain.csv", index_col="Unnamed: 0")
df.head()

## Exploratory Data Analysis

We begin by familiarizing ourselves with the dataset: inspecting its shape and checking for missing or duplicate values. We also use `info()` to verify the data type of each column.

In [ ]:
print(f"Dataset size: {df.shape}")
print(f"Missing values: {df.isna().sum().sum()}")
print(f"Duplicate values: {df.duplicated().sum()}")

In [ ]:
df.info()

The dataset contains no missing or duplicate values. Among the 22 features, all data types are correctly inferred except for the two temporal columns `trans_date_trans_time` and `dob`, which must be converted to datetime before any further processing.

In [ ]:
df["trans_date_trans_time"]=pd.to_datetime(df["trans_date_trans_time"])
df["dob"]=pd.to_datetime(df["dob"])

We begin the EDA by examining the target variable `is_fraud` to assess the distribution of fraudulent transactions.

In [ ]:
frauds=df["is_fraud"].value_counts()
fig,ax=plt.subplots()
ax.bar(frauds.index,frauds.values)
ax.set_xticks(frauds.index)
ax.set_title("Frauds distribution")
ax.set_xlabel(f"0: Not Fraud ({round(frauds[0]/frauds.sum()*100,2)}%), 1: Fraud ({round(frauds[1]/frauds.sum()*100,2)}%)")
ax.set_ylabel("Occurrences")

plt.tight_layout()
plt.show()

As expected, fraudulent transactions represent a negligible fraction of the dataset (~0.58%). This severe class imbalance means we are not dealing with a standard binary classification problem, but rather an anomaly detection task. We proceed by inspecting `trans_date_trans_time` to identify any temporal patterns in fraudulent activity.

In [ ]:
trans_hour=df["trans_date_trans_time"].dt.hour
fraud_hour=df["trans_date_trans_time"][df["is_fraud"]==1].dt.hour
trans_hour_counts=trans_hour.value_counts().sort_index()
fraud_hour_counts=fraud_hour.value_counts().sort_index()
perc_frauds=fraud_hour_counts/trans_hour_counts*100

fig,ax=plt.subplots(ncols=2, figsize=(10,5))
ax[0].bar(np.arange(24),fraud_hour_counts)
ax[0].set_xticks(np.arange(0,24,2))
ax[0].set_xlabel("Hour of the day")
ax[0].set_ylabel("Number of frauds")
ax[0].set_title("Fraud occurrences per hour")

ax[1].bar(np.arange(24),perc_frauds)
ax[1].set_xticks(np.arange(0,24,2))
ax[1].set_xlabel("Hour of the day")
ax[1].set_ylabel("% of frauds")
ax[1].set_title("% of frauds per hour")

plt.tight_layout()
plt.show()


The first important signal emerges clearly: fraud is strongly concentrated between 10 PM and 3 AM. This holds both in absolute count and as a percentage of total transactions per hour, ruling out any confound from varying transaction volumes throughout the day.

We further decompose the temporal dimension to check whether the day of the week, the day of the month, and the month of the year carry additional signal.

In [ ]:
fig, ax = plt.subplots(nrows=3, ncols=2,figsize=(10,8))

trans_dow = df["trans_date_trans_time"].dt.dayofweek
fraud_dow = df["trans_date_trans_time"][df["is_fraud"]==1].dt.dayofweek
trans_dow_counts = trans_dow.value_counts().sort_index()
fraud_dow_counts = fraud_dow.value_counts().sort_index()
perc_frauds_dow = (fraud_dow_counts / trans_dow_counts) * 100

dow = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

ax[0, 0].bar(trans_dow_counts.index, fraud_dow_counts)
ax[0, 0].set_xticks(range(7))
ax[0, 0].set_xticklabels(dow)
ax[0, 0].set_ylabel("Number of frauds")
ax[0, 0].set_title("Fraud occurrences per day of week")

ax[0, 1].bar(trans_dow_counts.index, perc_frauds_dow)
ax[0, 1].set_xticks(range(7))
ax[0, 1].set_xticklabels(dow)
ax[0, 1].set_ylabel("% of frauds")
ax[0, 1].set_title("% of frauds per day of week")

trans_dom = df["trans_date_trans_time"].dt.day
fraud_dom = df["trans_date_trans_time"][df["is_fraud"]==1].dt.day
trans_dom_counts = trans_dom.value_counts().sort_index()
fraud_dom_counts = fraud_dom.value_counts().sort_index()
perc_frauds_dom = (fraud_dom_counts / trans_dom_counts) * 100

ax[1, 0].bar(trans_dom_counts.index, fraud_dom_counts)
ax[1, 0].set_xticks(range(1, 32, 2))
ax[1, 0].set_xlabel("Day of month")
ax[1, 0].set_ylabel("Number of frauds")
ax[1, 0].set_title("Fraud occurrences per day of month")

ax[1, 1].bar(trans_dom_counts.index, perc_frauds_dom)
ax[1, 1].set_xticks(range(1, 32, 2))
ax[1, 1].set_xlabel("Day of month")
ax[1, 1].set_ylabel("% of frauds")
ax[1, 1].set_title("% of frauds per day of month")

trans_month = df["trans_date_trans_time"].dt.month
fraud_month = df["trans_date_trans_time"][df["is_fraud"] == 1].dt.month
trans_month_counts = trans_month.value_counts().sort_index()
fraud_month_counts = fraud_month.value_counts().sort_index()
perc_frauds_month = (fraud_month_counts / trans_month_counts) * 100

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']


ax[2,0].bar(fraud_month_counts.index, fraud_month_counts.values)
ax[2,0].set_xticks(range(1, 13))
ax[2,0].set_xticklabels(months)
ax[2,0].set_xlabel("Month")
ax[2,0].set_ylabel("Number of frauds")
ax[2,0].set_title("Fraud occurrences per month")

ax[2,1].bar(perc_frauds_month.index, perc_frauds_month.values)
ax[2,1].set_xticks(range(1, 13))
ax[2,1].set_xticklabels(months)
ax[2,1].set_xlabel("Month")
ax[2,1].set_ylabel("% of frauds")
ax[2,1].set_title("% of frauds per month")

plt.tight_layout()
plt.show()

Broadening the temporal granularity yields no additional signal. Neither the day of the week, the day of the month, nor the month of the year show a consistent pattern in the fraud rate. The hour of the day remains the only informative temporal feature and will be extracted during feature engineering.

In [ ]:
trans_cat = df["category"].value_counts()
fraud_cat = df[df["is_fraud"] == 1]["category"].value_counts()
perc_frauds_cat = (fraud_cat / trans_cat * 100).dropna().sort_values(ascending=True)
fraud_cat_sorted = fraud_cat.sort_values(ascending=True)

fig, ax = plt.subplots(ncols=2, figsize=(15, 7))

ax[0].barh(fraud_cat_sorted.index, fraud_cat_sorted.values)
ax[0].set_xlabel("Number of frauds")
ax[0].set_title("Top 10 categories by number of frauds")

ax[1].barh(perc_frauds_cat.index, perc_frauds_cat.values)
ax[1].set_xlabel("% of frauds")
ax[1].set_title("Top 10 categories by % of frauds")

plt.tight_layout()
plt.show()

A clear signal emerges: while the most represented categories in absolute terms are everyday ones such as groceries and shopping, the fraud rate is dominated by online transactions — those belonging to categories ending with the *_net* suffix.

Before moving to the next feature, we briefly inspect `merchant` to verify whether it carries any signal beyond what `category` already captures.

In [ ]:
trans_merch = df['merchant'].value_counts()
fraud_merch = df[df['is_fraud'] == 1]['merchant'].value_counts()

top20_abs = fraud_merch.sort_values(ascending=True).tail(10)
top20_perc = (fraud_merch / trans_merch * 100).dropna().sort_values(ascending=True).tail(10)

fig, ax = plt.subplots(ncols=2, figsize=(15, 7))

ax[0].barh(top20_abs.index, top20_abs.values)
ax[0].set_xlabel("Number of frauds")
ax[0].set_title("Top 10 merchants by number of frauds")

ax[1].barh(top20_perc.index, top20_perc.values)
ax[1].set_xlabel("% of frauds")
ax[1].set_title("Top 10 merchants by % of frauds")

plt.tight_layout()
plt.show()

The `merchant` feature has approximately 700 unique values. We limit the analysis to the top 10 merchants by absolute fraud count and by fraud rate, following the same dual-perspective approach used throughout the EDA.

No consistent signal emerges from the merchant-level analysis. The merchants with the highest fraud rates are those with very few total transactions — a statistical artefact rather than a genuine pattern. With ~700 unique values and no signal beyond what `category` already captures at cardinality 14, `merchant` will be dropped.

We now turn to the `amt` feature, which represents the transaction amount in dollars.

In [ ]:
print("Stats legit transactions")
print(df[df['is_fraud'] == 0]['amt'].describe())
print("\nStats frauds")
print(df[df['is_fraud'] == 1]['amt'].describe())

fig, ax = plt.subplots(ncols=2, figsize=(15, 6))

ax[0].boxplot([df[df["is_fraud"]==0]["amt"], df[df["is_fraud"]==1]["amt"]], tick_labels=["Not Fraud", "Fraud"])
ax[0].set_yscale("log") 
ax[0].set_ylabel("Amount ($, log scale)")
ax[0].set_title("Transactions amounts boxplot (log scale)")

ax[1].hist(df[(df["is_fraud"]==0) & (df["amt"] < 1400)]["amt"], bins=50, alpha=0.5, label="Not Fraud", density=True)
ax[1].hist(df[(df["is_fraud"]==1) & (df["amt"] < 1400)]["amt"], bins=50, alpha=0.5, label="Fraud", density=True)
ax[1].set_xlabel("Amount ($)")
ax[1].set_ylabel("Frauds density")
ax[1].set_title("Amounts distributions (Zoom < 1400$)")
ax[1].legend()

plt.tight_layout()
plt.show()

The printed statistics, boxplots, and empirical distributions converge on a clear picture: legitimate transactions are concentrated in the range of tens of dollars, with a long but thin tail of occasional large purchases. Fraudulent transactions show a distinctly different profile — fraudsters typically begin with small test charges to verify the card is active, then follow up with larger transactions in the hundreds of dollars.

We next examine `dob` to check whether the cardholder's age is associated with fraud likelihood.

In [ ]:
trans_yob = df["dob"].dt.year
fraud_yob = df[df["is_fraud"] == 1]["dob"].dt.year
trans_yob_counts = trans_yob.value_counts().sort_index()
fraud_yob_counts = fraud_yob.value_counts().sort_index()
perc_frauds_yob = (fraud_yob_counts / trans_yob_counts) * 100

fig, ax = plt.subplots(ncols=2, figsize=(15, 5))

ax[0].bar(fraud_yob_counts.index, fraud_yob_counts.values)
ax[0].set_xlabel("Year of birth")
ax[0].set_ylabel("Number of frauds")
ax[0].set_title("Number of frauds per year of birth")


ax[1].bar(perc_frauds_yob.index, perc_frauds_yob.values)
ax[1].set_xlabel("Year of birth")
ax[1].set_ylabel("% of frauds")
ax[1].set_yscale("log")
ax[1].set_title("% of frauds per year of birth")


plt.tight_layout()
plt.show()

In [ ]:
df[df["dob"].dt.year==1925][["cc_num","is_fraud"]].value_counts()

The absolute fraud count by birth year simply reflects the active population (roughly those born between 1960 and 2000), with no age-related pattern in the fraud rate. The spike at 1925 appears anomalous but is entirely explained by a single compromised card — confirmed by the lookup above. Age carries no predictive signal and `dob` will be dropped.

We now check whether the distance between the cardholder's home location (`lat`, `long`) and the merchant's location (`merch_lat`, `merch_long`) is informative.

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0 
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    d=R*c
    return d

dist_km = haversine_distance(df['lat'], df['long'], df['merch_lat'], df['merch_long'])

fig, ax = plt.subplots(ncols=2, figsize=(15, 6))

ax[0].boxplot([dist_km[df["is_fraud"]==0], dist_km[df["is_fraud"]==1]], tick_labels=["Not Fraud", "Fraud"])
ax[0].set_ylabel("Distance (km)")
ax[0].set_title("Distance user-commerce")

ax[1].hist(dist_km[df["is_fraud"]==0], bins=50, alpha=0.5, label="Not Fraud", density=True)
ax[1].hist(dist_km[df["is_fraud"]==1], bins=50, alpha=0.5, label="Fraud", density=True)
ax[1].set_xlabel("Distance (km)")
ax[1].set_ylabel("Frauds density")
ax[1].set_title("Distances distributions")
ax[1].legend()

plt.tight_layout()
plt.show()

We compute the great-circle distance between the cardholder's registered address and the merchant location using the Haversine formula. Both the boxplots and the empirical distributions are nearly identical for fraudulent and legitimate transactions, confirming that geographical distance carries no useful signal.

We now check whether the cardholder's `gender` is a relevant predictor.

In [ ]:
trans_gender = df['gender'].value_counts().sort_index()
fraud_gender = df[df['is_fraud']==1]['gender'].value_counts().sort_index()
perc_gender = (fraud_gender / trans_gender) * 100

fig, ax = plt.subplots(ncols=2, figsize=(14, 5))

ax[0].bar(fraud_gender.index, fraud_gender.values)
ax[0].set_xlabel("Gender")
ax[0].set_ylabel("Number of frauds")
ax[0].set_title("Number of frauds per gender")

ax[1].bar(perc_gender.index, perc_gender.values)
ax[1].set_xlabel("Gender")
ax[1].set_ylabel("% of frauds")
ax[1].set_title("% of frauds per gender")

plt.tight_layout()
plt.show()

Gender carries no predictive signal: the fraud rate is essentially identical across both groups.

We next examine `city_pop`, the population of the cardholder's city of residence.

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(14, 5))

ax[0].boxplot([df[df["is_fraud"]==0]["city_pop"], df[df["is_fraud"]==1]["city_pop"]], tick_labels=["Not Fraud", "Fraud"])
ax[0].set_yscale("log")
ax[0].set_ylabel("Population (Log Scale)")
ax[0].set_title("Population boxplots (Log Scale)")

bins = np.logspace(np.log10(df['city_pop'].min()), np.log10(df['city_pop'].max()), 50)
ax[1].hist(df[df["is_fraud"]==0]["city_pop"], bins=bins, alpha=0.5, label="Not Fraud", density=True)
ax[1].hist(df[df["is_fraud"]==1]["city_pop"], bins=bins, alpha=0.5, label="Fraud", density=True)
ax[1].set_xscale("log")
ax[1].set_xlabel("Population (Log Scale)")
ax[1].set_ylabel("Population density")
ax[1].set_title("Population densities distribtions (Log Scale)")
ax[1].legend()

plt.tight_layout()
plt.show()

The distributions of city population are nearly identical for fraudulent and legitimate transactions — no useful signal here either.

To complete the geographical analysis, we examine fraud rates by US state. We deliberately avoid analysing raw coordinates at finer granularity, as the high dimensionality would introduce noise rather than signal.

In [ ]:
trans_state = df['state'].value_counts()
fraud_state = df[df['is_fraud']==1]['state'].value_counts()

top10_abs = fraud_state.sort_values(ascending=False).head(10)
perc_state = (fraud_state / trans_state * 100).sort_values(ascending=False).head(10)

fig, ax = plt.subplots(ncols=2, figsize=(14, 5))

ax[0].bar(top10_abs.index, top10_abs.values)
ax[0].set_xlabel("State")
ax[0].set_ylabel("Number of frauds")
ax[0].set_title("Top 10 state per number of frauds")

ax[1].bar(perc_state.index, perc_state.values)
ax[1].set_xlabel("State")
ax[1].set_ylabel("% of frauds")
ax[1].set_title("Top 10 states per % of frauds")

plt.tight_layout()
plt.show()

In [ ]:
df[df['state']=='DE'][["cc_num","state","is_fraud"]].value_counts()

In absolute terms, no state shows a disproportionately high fraud count. Delaware (DE) appears to stand out in relative terms, but a closer look reveals that a single compromised card accounts for all fraudulent transactions in that state — the same statistical artefact encountered with the 1925 birth year. No geographical feature at any granularity carries actionable signal.

We conclude the EDA by analysing transaction velocity: the time elapsed between consecutive transactions on the same card.

In [ ]:
df_vel = df[['cc_num', 'unix_time', 'is_fraud']].sort_values(['cc_num', 'unix_time'])
df_vel['hours_since_last'] = df_vel.groupby('cc_num')['unix_time'].diff() / 3600

legit_intervals = df_vel[df_vel['is_fraud'] == 0]['hours_since_last'].dropna()
fraud_intervals = df_vel[df_vel['is_fraud'] == 1]['hours_since_last'].dropna()

clip_h = 100
fig, ax = plt.subplots(ncols=2, figsize=(15, 5))

ax[0].boxplot([legit_intervals, fraud_intervals], tick_labels=['Not Fraud', 'Fraud'])
ax[0].set_yscale('log')
ax[0].set_ylabel('Hours (log scale)')
ax[0].set_title('Time between consecutive transactions per card (log scale)')

ax[1].hist(legit_intervals.clip(upper=clip_h), bins=50, alpha=0.5, label='Not Fraud', density=True)
ax[1].hist(fraud_intervals.clip(upper=clip_h), bins=50, alpha=0.5, label='Fraud', density=True)
ax[1].set_xlabel(f'Hours since previous transaction on same card (clipped at {clip_h}h)')
ax[1].set_ylabel('Density')
ax[1].set_title('Transaction velocity per card')
ax[1].legend()

plt.tight_layout()
plt.show()

print(f"Median hours since last transaction — Not Fraud: {legit_intervals.median():.1f}h")
print(f"Median hours since last transaction — Fraud:     {fraud_intervals.median():.1f}h")

Fraudulent transactions occur significantly closer together in time than legitimate ones: the median interval drops from 4.6 hours for legitimate transactions to 1.4 hours for fraudulent ones. This is consistent with the typical fraud pattern — a burst of rapid charges once a card is stolen. The time elapsed since the last transaction on the same card (`hours_since_last_trans`) is a strong candidate for feature engineering.

The raw `amt` feature carries meaningful signal, but its discriminative power increases when conditioned on the purchase category. A transaction of $500 is unremarkable for `shopping_net` but anomalous for `gas_transport`. We engineer `amt_zscore_per_category` as the z-score of `amt` within each category, computed using the mean and standard deviation of legitimate transactions only — making it a measure of how unusual a given amount is relative to the typical spend in that category.

In [ ]:
cat_mean = df[df['is_fraud'] == 0].groupby('category')['amt'].mean()
cat_std  = df[df['is_fraud'] == 0].groupby('category')['amt'].std()

amt_z   = (df['amt'] - df['category'].map(cat_mean)) / (df['category'].map(cat_std) + 1e-6)
legit_z = amt_z[df['is_fraud'] == 0]
fraud_z = amt_z[df['is_fraud'] == 1]

clip_z = 10
fig, ax = plt.subplots(ncols=2, figsize=(14, 5))

ax[0].boxplot([legit_z.clip(-clip_z, clip_z), fraud_z.clip(-clip_z, clip_z)],
              tick_labels=['Not Fraud', 'Fraud'])
ax[0].set_ylabel('Amount z-score per category')
ax[0].set_title('amt_zscore_per_category')

ax[1].hist(legit_z.clip(-clip_z, clip_z), bins=60, alpha=0.5, label='Not Fraud', density=True)
ax[1].hist(fraud_z.clip(-clip_z, clip_z), bins=60, alpha=0.5, label='Fraud', density=True)
ax[1].set_xlabel('Z-score (clipped at ±10)')
ax[1].set_ylabel('Density')
ax[1].set_title('Distribution of amount z-score per category')
ax[1].legend()

plt.tight_layout()
plt.show()

print(f"Median   — Not Fraud: {legit_z.median():.2f}   Fraud: {fraud_z.median():.2f}")


`amt_zscore_per_category` normalises the amount relative to the category — but two cardholders in the same category can have very different spending habits. A $300 charge on `misc_net` may be typical for one cardholder and three standard deviations above their norm for another. We engineer `amt_zscore_per_card` as the z-score of `amt` relative to each cardholder's own historical mean and standard deviation, offering a complementary and orthogonal signal: how anomalous is this amount *for this specific card*.

In [ ]:
card_mean  = df.groupby('cc_num')['amt'].transform('mean')
card_std   = df.groupby('cc_num')['amt'].transform('std')
amt_z_card = (df['amt'] - card_mean) / (card_std + 1e-6)

legit_zc = amt_z_card[df['is_fraud'] == 0]
fraud_zc = amt_z_card[df['is_fraud'] == 1]

clip_z = 10
fig, ax = plt.subplots(ncols=2, figsize=(14, 5))

ax[0].boxplot([legit_zc.clip(-clip_z, clip_z), fraud_zc.clip(-clip_z, clip_z)],
              tick_labels=['Not Fraud', 'Fraud'])
ax[0].set_ylabel('Amount z-score per card')
ax[0].set_title('amt_zscore_per_card')

ax[1].hist(legit_zc.clip(-clip_z, clip_z), bins=60, alpha=0.5, label='Not Fraud', density=True)
ax[1].hist(fraud_zc.clip(-clip_z, clip_z), bins=60, alpha=0.5, label='Fraud', density=True)
ax[1].set_xlabel('Z-score (clipped at ±10)')
ax[1].set_ylabel('Density')
ax[1].set_title('Distribution of amount z-score per card')
ax[1].legend()

plt.tight_layout()
plt.show()

print(f"Median   — Not Fraud: {legit_zc.median():.2f}   Fraud: {fraud_zc.median():.2f}")

## Metrics evaluation function

In [ ]:
def evaluate_model(y_val, y_proba, model_name="Model", threshold=0.5):
    y_pred  = (y_proba >= threshold).astype(int)

    print(f"=== {model_name} ===")
    print(classification_report(y_val, y_pred, target_names=['Not Fraud', 'Fraud']))
    print(f"ROC-AUC : {roc_auc_score(y_val, y_proba):.4f}")
    print(f"PR-AUC  : {average_precision_score(y_val, y_proba):.4f}")

    fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(20, 12))

    ConfusionMatrixDisplay.from_predictions(y_val, y_pred, ax=ax[0, 0], display_labels=['Not Fraud', 'Fraud'])
    ax[0, 0].set_title(f'{model_name} — Confusion Matrix')

    RocCurveDisplay.from_predictions(y_val, y_proba, ax=ax[0, 1])
    ax[0, 1].set_title(f'{model_name} — ROC Curve')

    PrecisionRecallDisplay.from_predictions(y_val, y_proba, ax=ax[0, 2])
    ax[0, 2].set_title(f'{model_name} — Precision-Recall Curve')

    ax[1, 0].hist(y_proba[np.array(y_val) == 0], bins=100, alpha=0.5, label='Not Fraud', density=True)
    ax[1, 0].hist(y_proba[np.array(y_val) == 1], bins=100, alpha=0.5, label='Fraud', density=True)
    ax[1, 0].set_yscale('log')
    ax[1, 0].set_xlabel('Anomaly score')
    ax[1, 0].set_title(f'{model_name} — Score Distribution')
    ax[1, 0].legend()

    precision, recall, thresholds = precision_recall_curve(y_val, y_proba)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    ax[1, 1].plot(thresholds, precision[:-1], label='Precision')
    ax[1, 1].plot(thresholds, recall[:-1], label='Recall')
    ax[1, 1].plot(thresholds, f1[:-1], label='F1')
    ax[1, 1].set_xlabel('Threshold')
    ax[1, 1].set_title(f'{model_name} — Metrics vs Threshold')
    ax[1, 1].legend()

    ax[1, 2].axis('off')

    plt.tight_layout()
    plt.show()

## Fase 1 — Supervised Models

We establish the supervised ceiling with full label access. **Logistic Regression** serves as the linear baseline; **LightGBM** as the non-linear champion. Both models use the same hand-crafted feature set: transaction velocity, cyclical hour encoding, and card- and category-level amount z-scores.

This section answers a single question: *how well can a supervised model perform when fraud labels are abundant?* The answer sets the benchmark that Fase 3 will probe as labels are progressively removed.

## Supervised Baseline — Logistic Regression

Logistic Regression establishes the supervised baseline. With `class_weight='balanced'` the model reweights the minority class to compensate for the ~170:1 imbalance. Log-transforming `amt` and `hours_since_last_trans` linearises their right-skewed distributions, making them accessible to a linear decision boundary. Any non-linear method should comfortably surpass this baseline.

### Feature Engineering

In [ ]:
df_baseline = df.copy()
df_baseline = df_baseline.sort_values(['cc_num', 'unix_time'])

df_baseline['hours_since_last_trans'] = df_baseline.groupby('cc_num')['unix_time'].diff() / 3600
df_baseline['hours_since_last_trans'] = df_baseline['hours_since_last_trans'].fillna(
    df_baseline['hours_since_last_trans'].median()
)

_hour = df_baseline['trans_date_trans_time'].dt.hour
df_baseline['hour_sin'] = np.sin(2 * np.pi * _hour / 24)
df_baseline['hour_cos'] = np.cos(2 * np.pi * _hour / 24)

_cat_mean = df[df['is_fraud'] == 0].groupby('category', observed=True)['amt'].mean()
_cat_std  = df[df['is_fraud'] == 0].groupby('category', observed=True)['amt'].std()

df_baseline['amt_zscore_cat'] = (
    (df_baseline['amt'] - df_baseline['category'].map(_cat_mean))
    / (df_baseline['category'].map(_cat_std) + 1e-6)
)

_g = df_baseline.groupby('cc_num')['amt']
df_baseline['amt_zscore_card'] = (
    (df_baseline['amt'] - _g.transform(lambda x: x.expanding().mean().shift(1)))
    / (_g.transform(lambda x: x.expanding().std().shift(1)) + 1e-6)
).fillna(0)

df_baseline['amt'] = np.log1p(df_baseline['amt'])
df_baseline['hours_since_last_trans'] = np.log1p(df_baseline['hours_since_last_trans'])

cols_to_drop = ['trans_num', 'cc_num', 'first', 'last', 'gender',
                'street', 'city', 'state', 'zip', 'lat', 'long',
                'merch_lat', 'merch_long', 'city_pop', 'job',
                'merchant', 'unix_time', 'dob', 'trans_date_trans_time']
df_baseline = df_baseline.drop(columns=cols_to_drop).sort_index()


### Preprocessing

In [ ]:
X = df_baseline.drop(columns=['is_fraud'])
y = df_baseline['is_fraud']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, shuffle=False)

num_features = ['amt', 'hours_since_last_trans', 'hour_sin', 'hour_cos', 'amt_zscore_cat', 'amt_zscore_card']
cat_features = ['category']

preprocessor = ColumnTransformer([
    ('ohe', OneHotEncoder(handle_unknown='ignore'), cat_features),
    ('scaler', StandardScaler(), num_features)
])

pipe = Pipeline([
    ('pre', preprocessor),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42, n_jobs=-1))
])

pipe.fit(X_train, y_train)


### LogReg - Model evalutaion

In [ ]:
y_proba_lr = pipe.predict_proba(X_val)[:, 1]
evaluate_model(y_val, y_proba_lr, "Logistic Regression Baseline")

## Supervised Advanced — LightGBM

LightGBM is a gradient-boosted tree framework optimised for large tabular datasets. It captures non-linear feature interactions natively without requiring manual transformations — `hour` is kept raw rather than cyclically encoded since tree splits handle the 23→0 boundary automatically. The `scale_pos_weight` parameter rebalances the positive class in proportion to the training class ratio. Early stopping on the validation set prevents overfitting beyond the optimal tree count.

### Feature Engineering

In [ ]:
df_lgbm = df.copy()
df_lgbm = df_lgbm.sort_values(['cc_num', 'unix_time'])

df_lgbm['hours_since_last_trans'] = df_lgbm.groupby('cc_num')['unix_time'].diff() / 3600
df_lgbm['hours_since_last_trans'] = df_lgbm['hours_since_last_trans'].fillna(
    df_lgbm['hours_since_last_trans'].median()
)

df_lgbm['hour'] = df_lgbm['trans_date_trans_time'].dt.hour

_cat_mean = df[df['is_fraud'] == 0].groupby('category', observed=True)['amt'].mean()
_cat_std  = df[df['is_fraud'] == 0].groupby('category', observed=True)['amt'].std()

df_lgbm['amt_zscore_cat'] = (
    (df_lgbm['amt'] - df_lgbm['category'].map(_cat_mean))
    / (df_lgbm['category'].map(_cat_std) + 1e-6)
)

_g = df_lgbm.groupby('cc_num')['amt']
df_lgbm['amt_zscore_card'] = (
    (df_lgbm['amt'] - _g.transform(lambda x: x.expanding().mean().shift(1)))
    / (_g.transform(lambda x: x.expanding().std().shift(1)) + 1e-6)
).fillna(0)

cols_to_drop = ['trans_num', 'cc_num', 'first', 'last', 'gender',
                'street', 'city', 'state', 'zip', 'lat', 'long',
                'merch_lat', 'merch_long', 'city_pop', 'job',
                'merchant', 'unix_time', 'dob', 'trans_date_trans_time']
df_lgbm = df_lgbm.drop(columns=cols_to_drop)
df_lgbm['category'] = df_lgbm['category'].astype('category')  
df_lgbm = df_lgbm.sort_index()


### Preprocessing

In [ ]:
X_lgbm = df_lgbm.drop(columns=['is_fraud'])
y_lgbm = df_lgbm['is_fraud']

X_train_lgbm, X_val_lgbm, y_train_lgbm, y_val_lgbm = train_test_split(X_lgbm, y_lgbm, test_size=0.2, shuffle=False)

scale_pos = (y_train_lgbm == 0).sum() / (y_train_lgbm == 1).sum()

lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    scale_pos_weight=scale_pos,
    random_state=42,
    n_jobs=-1
)

lgbm.fit(
    X_train_lgbm, y_train_lgbm,
    categorical_feature=['category'],
    eval_set=[(X_val_lgbm, y_val_lgbm)],
    callbacks=[early_stopping(50, verbose=False), log_evaluation(100)]
)


### LightGBM - Model evalutation 

In [ ]:
y_proba_lgbm = lgbm.predict_proba(X_val_lgbm)[:, 1]
evaluate_model(y_val_lgbm, y_proba_lgbm, "LightGBM")

### Fase 1 — Summary

With full labels, the supervised approach dominates. **PR-AUC is the primary metric** throughout this project: on a ~0.58% fraud rate, ROC-AUC is inflated by the easy negatives, while PR-AUC measures precision-recall trade-offs directly on the minority class.

LightGBM establishes the supervised ceiling; Logistic Regression quantifies the cost of linearity. Both require fraud labels to train — the question Fase 2 asks is: *how far can label-free methods reach?* Fase 3 then finds the exact label budget at which supervised becomes preferable.

## Fase 2 — Unsupervised Models

Without fraud labels, models must detect anomalies by learning what *normal* looks like. We evaluate two approaches in increasing order of architectural complexity:

1. **Vanilla Autoencoder** *(Section 2.1)* — per-transaction MSE reconstruction, the simplest neural baseline
2. **LSTM-VAE** *(Section 2.3)* — explicit sequence modelling of each card's transaction history with a probabilistic latent space

**Design principle — deliberate feature asymmetry.** Supervised models in Fase 1 use the full feature pipeline including `amt_zscore_card`, the per-card expanding z-score, because fraud labels provide a safe anchor for history-dependent statistics. Here, without labels, that feature would pre-compute exactly what the LSTM-VAE is designed to learn from the sequence, removing the architectural motivation for temporal modelling. Both unsupervised models therefore operate on the same minimal feature set: transaction velocity, cyclical hour encoding, and category-level amount z-score — features that require no per-card historical aggregation.

This asymmetry is not a concession — it is the thesis premise. Fase 3 makes the trade-off explicit by sweeping label budgets and locating the crossover point at which supervised becomes preferable.

### Feature Engineering and Preprocessing

Base per-transaction features shared by both models in Fase 2. `amt_zscore_card` is intentionally excluded: it pre-computes a per-card spending norm that the LSTM-VAE is designed to discover from the sequence itself. Including it would make the sequential encoder redundant. The remaining features — transaction velocity (`hours_since_last_trans`), cyclical hour encoding (`hour_sin`, `hour_cos`), and category-level amount z-score (`amt_zscore_cat`) — require no per-card aggregation and are safe in the unsupervised setting.

In [ ]:
df_ae = df.copy()
df_ae = df_ae.sort_values(['cc_num', 'unix_time'])

df_ae['hours_since_last_trans'] = df_ae.groupby('cc_num')['unix_time'].diff() / 3600
df_ae['hours_since_last_trans'] = df_ae['hours_since_last_trans'].fillna(
    df_ae['hours_since_last_trans'].median()
)

_hour = df_ae['trans_date_trans_time'].dt.hour
df_ae['hour_sin'] = np.sin(2 * np.pi * _hour / 24)
df_ae['hour_cos'] = np.cos(2 * np.pi * _hour / 24)

_cat_mean = df[df['is_fraud'] == 0].groupby('category', observed=True)['amt'].mean()
_cat_std  = df[df['is_fraud'] == 0].groupby('category', observed=True)['amt'].std()

df_ae['amt_zscore_cat'] = (
    (df_ae['amt'] - df_ae['category'].map(_cat_mean))
    / (df_ae['category'].map(_cat_std) + 1e-6)
)

cols_to_drop = [
    'trans_num', 'cc_num', 'first', 'last', 'gender',
    'street', 'city', 'state', 'zip', 'lat', 'long',
    'merch_lat', 'merch_long', 'city_pop', 'job',
    'merchant', 'unix_time', 'dob', 'trans_date_trans_time'
]
df_ae = df_ae.drop(columns=cols_to_drop).sort_index()

### Preprocessing

In [ ]:
NUM_FEATURES = ['amt', 'hours_since_last_trans', 'hour_sin', 'hour_cos', 'amt_zscore_cat']
CAT_FEATURES = ['category']
EMB_DIMS     = {'category': 4}

X_ae = df_ae.drop(columns=['is_fraud'])
y_ae = df_ae['is_fraud']

X_train_ae, X_val_ae, y_train_ae, y_val_ae = train_test_split(
    X_ae, y_ae, test_size=0.2, shuffle=False
)
X_train_normal = X_train_ae[y_train_ae == 0].copy()

qt_ae = QuantileTransformer(output_distribution='normal', n_quantiles=1000, random_state=42)
qt_ae.fit(X_train_normal[NUM_FEATURES])

cat_vocabs      = {}
cat_vocab_sizes = {}
for col in CAT_FEATURES:
    vals = sorted(X_train_normal[col].dropna().unique())
    cat_vocabs[col]      = {v: i + 1 for i, v in enumerate(vals)}
    cat_vocab_sizes[col] = len(vals) + 1

def encode_cats(df_sub, vocabs, cols):
    arr = np.zeros((len(df_sub), len(cols)), dtype=np.int64)
    for j, col in enumerate(cols):
        arr[:, j] = df_sub[col].map(vocabs[col]).fillna(0).astype(int).values
    return arr

X_train_num = qt_ae.transform(X_train_normal[NUM_FEATURES]).astype(np.float32)
X_val_num   = qt_ae.transform(X_val_ae[NUM_FEATURES]).astype(np.float32)
X_train_cat = encode_cats(X_train_normal, cat_vocabs, CAT_FEATURES)
X_val_cat   = encode_cats(X_val_ae,       cat_vocabs, CAT_FEATURES)

### 2.1 — Vanilla Autoencoder

The simplest neural baseline: a symmetric encoder-decoder that compresses the five continuous transaction features into a low-dimensional latent space and reconstructs them with **MSE**. Trained on normal transactions only, it flags as anomalous any transaction it reconstructs poorly.

No categorical features, no probabilistic loss, no temporal context — a deliberate stripped-down baseline that isolates the reconstruction idea from everything else. Contrast with Section 2.3, where the encoder becomes an LSTM and the loss becomes a proper ELBO.

#### Architecture

Symmetric encoder-decoder on the 5 continuous features. Anomaly score = mean squared reconstruction error per transaction.

In [ ]:
class VanillaAE(nn.Module):
    def __init__(self, n_num, latent_dim=8, hidden_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_num, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, n_num),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

#### Training

MSE reconstruction loss on normal transactions only. No noise injection — clean baseline.

In [ ]:
torch.manual_seed(42)
EPOCHS = 30

X_tr_num_t  = torch.FloatTensor(X_train_num)
X_val_num_t = torch.FloatTensor(X_val_num)
y_val_np    = y_val_ae.values

loader = DataLoader(TensorDataset(X_tr_num_t), batch_size=256, shuffle=True)

model_ae  = VanillaAE(n_num=len(NUM_FEATURES))
optimizer = optim.Adam(model_ae.parameters(), lr=1e-3)

train_losses = []
model_ae.train()
for epoch in range(EPOCHS):
    epoch_loss = []
    for (x_num_b,) in loader:
        x_hat = model_ae(x_num_b)
        loss  = F.mse_loss(x_hat, x_num_b)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss.append(loss.item())
    train_losses.append(np.mean(epoch_loss))
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/{EPOCHS} — loss: {train_losses[-1]:.6f}")

plt.plot(train_losses)
plt.xlabel('Epoch'); plt.ylabel('MSE Loss'); plt.title('Vanilla AE — Training Loss')
plt.show()

#### Evaluation

In [ ]:
def threshold_from_normal_scores(scores_normal, percentile=99.5):
    return float(np.percentile(scores_normal, percentile))

model_ae.eval()
with torch.no_grad():
    scores_ae_train = F.mse_loss(model_ae(X_tr_num_t), X_tr_num_t, reduction='none').mean(dim=1).numpy()
    scores_ae_val   = F.mse_loss(model_ae(X_val_num_t), X_val_num_t, reduction='none').mean(dim=1).numpy()

thr_ae = threshold_from_normal_scores(scores_ae_train, percentile=99.5)
evaluate_model(y_val_np, scores_ae_val, "Vanilla AE", threshold=thr_ae)

#### Vanilla AE — Results

The Vanilla AE reconstructs each transaction independently with MSE. Without categorical context, probabilistic loss, or any notion of card history, its anomaly score reflects only how unusual the raw feature values are in isolation — a necessary baseline, but a deliberately weak one.

Section 2.3 replaces this per-transaction view with an LSTM encoder that processes each card's full transaction history, and upgrades the loss from MSE to a proper probabilistic ELBO.

### 2.3 — LSTM-VAE

Unlike the Vanilla AE, which reconstructs each transaction independently, the LSTM-VAE processes each card's **transaction history as a sequence**. The LSTM encoder compresses a window of `SEQ_LEN` recent transactions into a stochastic latent vector (`μ_z`, `σ_z`); the MLP decoder reconstructs the last (most recent) transaction conditioned on a sample `z`.

Both models in Fase 2 operate on the same 5-feature raw input — without `amt_zscore_card`. The LSTM is expected to learn per-card spending norms directly from the transaction sequence; providing a pre-computed card-level z-score would pre-empt exactly what the sequential encoder is designed to capture.

Training objective is the **ELBO**: reconstruction NLL of the target transaction + β · KL(q(z | x_seq) ‖ N(0, I)). β = 0.1 down-weights the KL term to prevent posterior collapse — a failure mode where the encoder ignores the input and the latent distribution collapses to the prior, resulting in uniformly poor anomaly scores.

**Anomaly score** = reconstruction NLL of the last transaction conditioned on encoded card history. A transaction easy to predict from its card's recent pattern scores low; one that breaks it scores high.

In [ ]:
SEQ_LEN = 10

# Recover card + time metadata (cc_num dropped in FE; original df index preserved)
_meta = df.loc[df_ae.index, ['cc_num', 'unix_time']].copy()
_meta['_row'] = np.arange(len(_meta))   # positional index in X_ae row order

# Quantile-transform and encode the full X_ae for window construction
_all_num = qt_ae.transform(X_ae[NUM_FEATURES]).astype(np.float32)
_all_cat = encode_cats(X_ae, cat_vocabs, CAT_FEATURES)

# Build padded windows: each row gets a SEQ_LEN-length window of same-card transactions
seq_num_all = np.zeros((len(_meta), SEQ_LEN, len(NUM_FEATURES)), dtype=np.float32)
seq_cat_all = np.zeros((len(_meta), SEQ_LEN, len(CAT_FEATURES)), dtype=np.int64)

for cc, grp in _meta.sort_values(['cc_num', 'unix_time']).groupby('cc_num', sort=False):
    rows = grp['_row'].values
    for k, pos in enumerate(rows):
        start  = max(0, k - SEQ_LEN + 1)
        window = rows[start:k+1]
        offset = SEQ_LEN - len(window)
        seq_num_all[pos, offset:] = _all_num[window]
        seq_cat_all[pos, offset:] = _all_cat[window]

# Align with train/val split (shuffle=False → first n_tr rows = train)
n_tr           = len(X_train_ae)
_tr_normal_pos = np.where(y_ae.values[:n_tr] == 0)[0]

seq_num_tr_t  = torch.FloatTensor(seq_num_all[_tr_normal_pos])
seq_cat_tr_t  = torch.LongTensor(seq_cat_all[_tr_normal_pos])
seq_num_val_t = torch.FloatTensor(seq_num_all[n_tr:])
seq_cat_val_t = torch.LongTensor(seq_cat_all[n_tr:])

#### Architecture and Loss

The LSTM reads the transaction window and outputs a final hidden state; two linear heads project it to `μ_z` and `log σ_z`. The MLP decoder reconstructs the last transaction from the sampled latent `z`.

**ELBO** = E_q[log p(x_last | z)] − β · KL(q(z | x_seq) ‖ N(0, I))

Reconstruction uses Gaussian NLL for the five continuous features and cross-entropy for `category`, weighted by `CONT_WEIGHT`.

In [ ]:
def mixed_nll_loss(x_num, x_cat, mu, log_sigma, cat_logits, cont_weight):
    """Gaussian NLL for continuous features + cross-entropy for categorical."""
    sigma    = log_sigma.exp()
    nll_cont = (log_sigma + 0.5 * ((x_num - mu) / sigma) ** 2).sum(dim=1).mean()
    nll_cat  = torch.stack([
        F.cross_entropy(cat_logits[j], x_cat[:, j], reduction='none')
        for j in range(x_cat.shape[1])
    ], dim=1).sum(dim=1).mean()
    return cont_weight * nll_cont + (1 - cont_weight) * nll_cat


class LSTMVAE(nn.Module):
    def __init__(self, vocab_sizes, emb_dims, cat_cols, n_num,
                 hidden_dim=64, latent_dim=16):
        super().__init__()
        self.cat_cols = cat_cols
        self.n_num    = n_num

        self.embeddings = nn.ModuleList([
            nn.Embedding(vocab_sizes[col], emb_dims[col]) for col in cat_cols
        ])
        input_dim = n_num + sum(emb_dims[col] for col in cat_cols)

        # LSTM encoder: processes the full window → final hidden state → (μ_z, log σ_z)
        self.lstm         = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc_mu        = nn.Linear(hidden_dim, latent_dim)
        self.fc_log_sigma = nn.Linear(hidden_dim, latent_dim)

        # MLP decoder: z → reconstruction of the last transaction in the window
        self.decoder   = nn.Sequential(nn.Linear(latent_dim, hidden_dim), nn.ReLU())
        self.cont_head = nn.Linear(hidden_dim, n_num * 2)       # μ_x and log σ_x per feature
        self.cat_heads = nn.ModuleList([
            nn.Linear(hidden_dim, vocab_sizes[col]) for col in cat_cols
        ])

    def encode(self, x_num_seq, x_cat_seq):
        embs = [self.embeddings[i](x_cat_seq[:, :, i]) for i in range(len(self.cat_cols))]
        x    = torch.cat(embs + [x_num_seq], dim=2)
        _, (h, _) = self.lstm(x)
        h = h.squeeze(0)
        return self.fc_mu(h), self.fc_log_sigma(h).clamp(-4, 2)

    def decode(self, z):
        h         = self.decoder(z)
        cont      = self.cont_head(h)
        mu        = cont[:, :self.n_num]
        log_sigma = cont[:, self.n_num:].clamp(-5, 2)
        return mu, log_sigma, [head(h) for head in self.cat_heads]

    def forward(self, x_num_seq, x_cat_seq):
        mu_z, log_sigma_z = self.encode(x_num_seq, x_cat_seq)
        z = mu_z + torch.randn_like(mu_z) * log_sigma_z.exp() if self.training else mu_z
        mu_x, log_sigma_x, cat_logits = self.decode(z)
        return mu_x, log_sigma_x, cat_logits, mu_z, log_sigma_z

#### Training

`CONT_WEIGHT=0.75`, `β=0.1` (down-weighted KL to prevent posterior collapse). The model sees windows of `SEQ_LEN=10` normal transactions and learns to reconstruct the last one from the encoded sequence. Reconstruction NLL and KL divergence are logged separately every 10 epochs — a diverging KL with flat reconstruction would indicate posterior collapse.

In [ ]:
torch.manual_seed(42)
CONT_WEIGHT   = 0.75
BETA          = 0.1
EPOCHS_VAE    = 60

loader_vae = DataLoader(
    TensorDataset(seq_num_tr_t, seq_cat_tr_t),
    batch_size=256, shuffle=True
)

model_vae     = LSTMVAE(vocab_sizes=cat_vocab_sizes, emb_dims=EMB_DIMS,
                        cat_cols=CAT_FEATURES, n_num=len(NUM_FEATURES))
optimizer_vae = optim.Adam(model_vae.parameters(), lr=1e-3)

train_losses_vae = []
model_vae.train()
for epoch in range(EPOCHS_VAE):
    epoch_loss, epoch_recon, epoch_kl = [], [], []
    for x_num_seq, x_cat_seq in loader_vae:
        x_num_last = x_num_seq[:, -1, :]
        x_cat_last = x_cat_seq[:, -1, :]
        mu_x, log_sigma_x, cat_logits, mu_z, log_sigma_z = model_vae(x_num_seq, x_cat_seq)
        recon = mixed_nll_loss(x_num_last, x_cat_last, mu_x, log_sigma_x, cat_logits, CONT_WEIGHT)
        kl    = 0.5 * ((2*log_sigma_z).exp() + mu_z.pow(2) - 1 - 2*log_sigma_z).sum(dim=1).mean()
        loss  = recon + BETA * kl
        optimizer_vae.zero_grad(); loss.backward(); optimizer_vae.step()
        epoch_loss.append(loss.item())
        epoch_recon.append(recon.item())
        epoch_kl.append(kl.item())
    train_losses_vae.append(np.mean(epoch_loss))
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:2d}/{EPOCHS_VAE} — loss: {train_losses_vae[-1]:.4f}  "
              f"recon: {np.mean(epoch_recon):.4f}  kl: {np.mean(epoch_kl):.4f}")

plt.plot(train_losses_vae)
plt.xlabel('Epoch'); plt.ylabel('ELBO'); plt.title('LSTM-VAE — Training Loss')
plt.show()

#### Evaluation

Anomaly score = reconstruction NLL of the last transaction in the window, conditioned on the encoded card history (`z` = posterior mean at inference for stability). Threshold at the 99.5th percentile of normal-transaction training scores.

In [ ]:
def score_lstm_vae(model, seq_num_t, seq_cat_t, cont_weight, batch_size=2048):
    model.eval()
    scores = []
    with torch.no_grad():
        for i in range(0, len(seq_num_t), batch_size):
            sn, sc  = seq_num_t[i:i+batch_size], seq_cat_t[i:i+batch_size]
            xn_last = sn[:, -1, :]
            xc_last = sc[:, -1, :]
            mu_x, log_sigma_x, cat_logits, _, _ = model(sn, sc)
            sigma_x = log_sigma_x.exp()
            s_cont  = (log_sigma_x + 0.5 * ((xn_last - mu_x) / sigma_x) ** 2).sum(dim=1)
            s_cat   = torch.stack([
                F.cross_entropy(cat_logits[j], xc_last[:, j], reduction='none')
                for j in range(xc_last.shape[1])
            ], dim=1).sum(dim=1)
            scores.append((cont_weight * s_cont + (1 - cont_weight) * s_cat).cpu())
    return torch.cat(scores).numpy()

scores_vae_tr  = score_lstm_vae(model_vae, seq_num_tr_t, seq_cat_tr_t, CONT_WEIGHT)
scores_vae_val = score_lstm_vae(model_vae, seq_num_val_t, seq_cat_val_t, CONT_WEIGHT)

thr_vae = threshold_from_normal_scores(scores_vae_tr, percentile=99.5)
evaluate_model(y_val_ae.values, scores_vae_val, "LSTM-VAE", threshold=thr_vae)